# boréal marché

github repository: [temelal/boreal-reco-temel-melal](https://github.com/temelal/boreal-reco-temel-melal)

i use the full transaction file to find basket rules and save them for a small shop application.
the application reads the saved files. the expensive mining is done here.

In [1]:
import gc
import json
import platform
import re
import subprocess
import sys
import tempfile
import time
import unicodedata
from datetime import datetime
from importlib.metadata import version
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "data" / "transactions.zip").exists()
workspace = tempfile.TemporaryDirectory(prefix="boreal-mining-")
scratch = Path(workspace.name)
products = pd.read_csv(root / "data" / "products.csv")
items = products["Item"].tolist()
names = dict(zip(products["Item"], products["Item_EN"].str.lower()))
minimum_confidence = 0.2
display(pd.DataFrame({
    "package": ["python", "numpy", "pandas", "mlxtend"],
    "version": [platform.python_version(), version("numpy"), version("pandas"), version("mlxtend")]
}))

,package,version
0,python,3.12.14
1,numpy,2.5.4
2,pandas,3.1.0
3,mlxtend,0.25.0


## phase 1: load and prepare

the file is in long format. transactionid is the basket key, not customerid.
the supplied archive was called transactionsdata.zip, i copied it as transactions.zip.
the brief says item_fr but the catalogue really has item_en. i use item for the original french key,
and item_en for the english display name.

In [2]:
started = time.perf_counter()
tx = pd.read_csv(
    root / "data" / "transactions.zip",
    dtype={"TransactionID": "int32", "CustomerID": "int32", "Date": "category", "Item": "category"}
)
load_seconds = time.perf_counter() - started
assert tx.isna().sum().sum() == 0
assert tx["TransactionID"].nunique() == 1_000_000
assert len(tx) == 8_379_837
assert len(products) == tx["Item"].nunique() == 98
assert set(tx["Item"]) == set(items)
duplicate_rows = int(tx.duplicated(["TransactionID", "Item"]).sum())
assert duplicate_rows == 0
date_check = tx.groupby("TransactionID", sort=False, observed=True)["Date"].nunique()
customer_check = tx.groupby("TransactionID", sort=False)["CustomerID"].nunique()
assert date_check.eq(1).all() and customer_check.eq(1).all()
profile = {
    "rows": len(tx),
    "baskets": tx["TransactionID"].nunique(),
    "products": len(products),
    "mean_basket_size": len(tx) / tx["TransactionID"].nunique(),
    "duplicate_basket_items": duplicate_rows,
    "missing_values": int(tx.isna().sum().sum()),
    "first_date": str(min(tx["Date"].cat.categories)),
    "last_date": str(max(tx["Date"].cat.categories)),
    "load_seconds": load_seconds,
    "long_memory_mib": float(tx.memory_usage(deep=True).sum() / 1024**2)
}
display(pd.DataFrame([profile]).T.rename(columns={0: "value"}))

,value
rows,8379837
baskets,1000000
products,98
mean_basket_size,8.379837
duplicate_basket_items,0
missing_values,0
first_date,2026-01-01
last_date,2026-12-31
load_seconds,0.932122
long_memory_mib,87.926261


### making the wide table

i factorize the basket ids and use the catalogue order for the product codes.
then i mark the matching positions in one boolean array. repeated scans would still give one true value.
this avoids making an 8.4 million by 98 dummy table first.
the matrix memory below is the array storage; it is different from the whole process peak memory.

In [3]:
started = time.perf_counter()
basket_codes, basket_ids = pd.factorize(tx["TransactionID"], sort=True)
product_codes = pd.Categorical(tx["Item"], categories=items).codes
assert product_codes.min() >= 0
values = np.zeros((len(basket_ids), len(items)), dtype=bool)
values[basket_codes, product_codes] = True
basket = pd.DataFrame(values, columns=items, copy=False)
wide_seconds = time.perf_counter() - started
profile["wide_seconds"] = wide_seconds
profile["matrix_mib"] = values.nbytes / 1024**2
profile["dataframe_mib"] = basket.memory_usage(deep=True).sum() / 1024**2
assert basket.shape == (1_000_000, 98)
assert all(dtype == bool for dtype in basket.dtypes)
assert int(values.sum()) == len(tx)
display(pd.DataFrame([{
    "baskets": basket.shape[0],
    "products": basket.shape[1],
    "wide_seconds": round(wide_seconds, 4),
    "matrix_mib": round(profile["matrix_mib"], 3),
    "dataframe_mib": round(profile["dataframe_mib"], 3)
}]))
item_counts = basket.sum().astype(int)
item_supports = item_counts / len(basket)
frequency = pd.DataFrame({
    "product": [names[item] for item in items],
    "baskets": item_counts.to_numpy(),
    "support": item_supports.to_numpy()
}).sort_values(["baskets", "product"], ascending=[False, True])
print("the 10 most frequent products")
display(frequency.head(10).reset_index(drop=True))
print("the 5 rarest products")
display(frequency.tail(5).sort_values("baskets").reset_index(drop=True))

,baskets,products,wide_seconds,matrix_mib,dataframe_mib
0,1000000,98,0.0938,93.46,93.46


the 10 most frequent products


,product,baskets,support
0,reusable bag,900026,0.900026
1,cookies,161439,0.161439
2,mixed nuts,155189,0.155189
3,peanut butter,153726,0.153726
4,beer,150468,0.150468
5,dish soap,149820,0.149820
6,cheddar cheese,143457,0.143457
7,charcoal,140561,0.140561
8,toilet paper,139758,0.139758
9,cereal,139198,0.139198


the 5 rarest products


,product,baskets,support
0,foie gras,756,0.000756
1,aged balsamic vinegar,779,0.000779
2,caviar,810,0.000810
3,truffle oil,817,0.000817
4,saffron,818,0.000818


### my prediction before mining

the reusable bag appears in about 90 percent of baskets. i expect many rules ending in this bag to have
high confidence, even when the basket product tells us very little about buying a bag.
their lift should be near one. confidence alone can make these rules look more useful than they are.
the five speciality products have less than 0.1 percent support, so even the lowest required threshold
will remove them before any rule can be made.

### duplicate product audit and decision

i check every french and english catalogue name after normalising case, accents, punctuation and spaces.
there is no exact normalised name duplicate. the two entries sharing the baguette word are the pair
i investigate for the duplicate-product question. i keep the rustic modifier because it describes a variant.
the file has no barcode or stock id, so a name match alone cannot prove one physical product identity.

my decision is to keep the two catalogue records separate in the main model and the application.
below i compare their prices, seasons and basket links. i also run a separate 97-column merge experiment
to measure what would change if the two names were aliases. a basket containing both must count only once.

In [4]:
def normalise_name(value):
    text = unicodedata.normalize("NFKD", str(value).casefold())
    text = "".join(letter for letter in text if not unicodedata.combining(letter))
    return " ".join(re.sub(r"[\W_]+", " ", text).split())

def duplicate_name_groups(column):
    groups = {}
    for row in products.to_dict("records"):
        groups.setdefault(normalise_name(row[column]), []).append(row["Item"])
    return [group for group in groups.values() if len(group) > 1]

french_duplicates = duplicate_name_groups("Item")
english_duplicates = duplicate_name_groups("Item_EN")
pair_items = [item for item in items if "baguette" in normalise_name(item).split()]
assert set(pair_items) == {"Baguette", "Pain baguette rustique"}
print("full catalogue check, keeping product modifiers")
display(pd.DataFrame([
    {"name_field": "french item", "records": len(products), "normalised_duplicate_groups": len(french_duplicates)},
    {"name_field": "english item", "records": len(products), "normalised_duplicate_groups": len(english_duplicates)}
]))
baguette = basket["Baguette"].to_numpy()
rustic = basket["Pain baguette rustique"].to_numpy()
merged_baguette = baguette | rustic
pair_records = [
    {
        "item": row["Item"],
        "item_en": row["Item_EN"].lower(),
        "category": row["Category"].lower(),
        "price": float(row["Price"]),
        "baskets": int(item_counts[row["Item"]]),
        "support": float(item_supports[row["Item"]])
    }
    for row in products[products["Item"].isin(pair_items)].to_dict("records")
]
display(pd.DataFrame(pair_records))
duplicate_audit = {
    "normalised_item_duplicate_groups": french_duplicates,
    "normalised_english_duplicate_groups": english_duplicates,
    "pair": pair_records,
    "overlap_baskets": int((baguette & rustic).sum()),
    "union_baskets": int(merged_baguette.sum()),
    "naive_sum": int(baguette.sum() + rustic.sum()),
    "raw_products": len(items),
    "merged_products": len(items) - 1,
    "raw_presences": int(values.sum()),
    "merged_presences": int(values.sum() - (baguette & rustic).sum()),
    "decision": "keep the two catalogue records separate",
    "exact_identity_verified": False
}
display(pd.DataFrame([{
    "regular_baskets": int(baguette.sum()),
    "rustic_baskets": int(rustic.sum()),
    "overlap": duplicate_audit["overlap_baskets"],
    "correct_or_union": duplicate_audit["union_baskets"],
    "incorrect_sum": duplicate_audit["naive_sum"]
}]))
dates = (
    tx.drop_duplicates("TransactionID")
    .set_index("TransactionID")["Date"]
    .astype(str)
    .reindex(basket_ids)
)
months = dates.str.slice(5, 7).astype("uint8").to_numpy()

def bread_link(left, consequent):
    right = basket[consequent].to_numpy()
    support = float((left & right).mean())
    confidence = float((left & right).sum() / left.sum())
    lift = float(confidence / right.mean())
    return support, confidence, lift

profiles = []
for item, left in [("Baguette", baguette), ("Pain baguette rustique", rustic)]:
    raclette = bread_link(left, "Fromage à raclette")
    fondue = bread_link(left, "Bouillon à fondue")
    profiles.append({
        "product": names[item],
        "summer_support": float(left[np.isin(months, [6, 7, 8])].mean()),
        "winter_support": float(left[np.isin(months, [12, 1, 2])].mean()),
        "raclette_confidence": raclette[1],
        "raclette_lift": raclette[2],
        "fondue_confidence": fondue[1],
        "fondue_lift": fondue[2]
    })
duplicate_audit["profiles"] = profiles
display(pd.DataFrame(profiles))
duplicate_rule_comparison = []
for representation, label, left in [
    ("raw regular", "baguette", baguette),
    ("raw rustic", "rustic baguette", rustic),
    ("merged family", "baguette family", merged_baguette)
]:
    for consequent in ["Fromage à raclette", "Bouillon à fondue"]:
        support, confidence, lift = bread_link(left, consequent)
        duplicate_rule_comparison.append({
            "representation": representation,
            "antecedent": label,
            "consequent": names[consequent],
            "support": support,
            "confidence": confidence,
            "lift": lift,
            "passes_confidence": confidence >= minimum_confidence
        })
print("direct rule metrics, including rules below the confidence threshold")
display(pd.DataFrame(duplicate_rule_comparison))
np.save(scratch / "annual.npy", values)
(scratch / "items.json").write_text(json.dumps(items, ensure_ascii=False), encoding="utf-8")
del tx, basket_codes, product_codes, dates, date_check, customer_check, baguette, rustic, merged_baguette, left
gc.collect()
print("the prediction above is recorded before the mining section in this notebook")

full catalogue check, keeping product modifiers


,name_field,records,normalised_duplicate_groups
0,french item,98,0
1,english item,98,0


,item,item_en,category,price,baskets,support
0,Baguette,baguette,boulangerie,2.99,69400,0.069400
1,Pain baguette rustique,rustic baguette,boulangerie,3.99,42219,0.042219


,regular_baskets,rustic_baskets,overlap,correct_or_union,incorrect_sum
0,69400,42219,3009,108610,111619


,product,summer_support,winter_support,raclette_confidence,raclette_lift,fondue_confidence,fondue_lift
0,baguette,0.069607,0.069334,0.068199,0.995487,0.047075,0.974737
1,rustic baguette,0.021626,0.069237,0.654374,9.551784,0.409981,8.489104


direct rule metrics, including rules below the confidence threshold


,representation,antecedent,consequent,support,confidence,lift,passes_confidence
0,raw regular,baguette,raclette cheese,0.004733,0.068199,0.995487,False
1,raw regular,baguette,fondue broth,0.003267,0.047075,0.974737,False
2,raw rustic,rustic baguette,raclette cheese,0.027627,0.654374,9.551784,True
3,raw rustic,rustic baguette,fondue broth,0.017309,0.409981,8.489104,True
4,merged family,baguette family,raclette cheese,0.030420,0.280085,4.088350,True
5,merged family,baguette family,fondue broth,0.019402,0.178639,3.698916,False


the prediction above is recorded before the mining section in this notebook


## phase 2: apriori experiments

i use only mlxtend apriori. all four runs use the full million baskets, a confidence threshold of 0.20,
and no maximum itemset length. each run is in a fresh process so its peak memory is measured separately.
time starts after the worker loads the matrix and covers mining, rule generation and conversion to records.
peak memory is the operating system high-water rss for the worker, including its loaded data and libraries.

first i try the default low_memory=false. i allow one extra gib of virtual address space above the loaded
worker, to observe the allocation error without putting the computer under memory pressure.
the same limit is used for the successful low-memory runs. this is a controlled memory-error experiment,
not a claim that this 64 gib computer has only one gib of physical ram.

In [5]:
def run_mining(label, path, support, low_memory, item_path=None):
    output = scratch / f"{label}.json"
    command = [
        sys.executable, str(root / "notebooks" / "mining_worker.py"),
        "--basket", str(path), "--items", str(item_path or scratch / "items.json"),
        "--output", str(output), "--min-support", str(support),
        "--min-confidence", str(minimum_confidence), "--extra-memory-mib", "1024"
    ]
    if low_memory:
        command.append("--low-memory")
    completed = subprocess.run(command, capture_output=True, text=True, timeout=1800)
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr)
    result = json.loads(output.read_text(encoding="utf-8"))
    print(f"{label}: {result['outcome']}, {result['elapsed_seconds']:.3f} seconds, {result['peak_memory_mib']:.1f} mib peak")
    return result

default_run = run_mining("default", scratch / "annual.npy", 0.05, False)
print(default_run.get("error", "the default run completed on this environment").lower())
experiments = [
    run_mining(f"annual_{support}", scratch / "annual.npy", support, True)
    for support in [0.05, 0.02, 0.01, 0.005]
]
assert all(run["outcome"] == "completed" for run in experiments)
benchmark = pd.DataFrame([{
    "min_support": run["min_support"],
    "itemsets": run["n_itemsets"],
    "rules": run["n_rules"],
    "time_seconds": round(run["elapsed_seconds"], 3),
    "peak_memory_mib": round(run["peak_memory_mib"], 3),
    "outcome": run["outcome"]
} for run in experiments])
display(benchmark)

default: memoryerror, 0.063 seconds, 201.3 mib peak
unable to allocate 4.76 gib for an array with shape (2556, 2, 1000000) and data type bool


annual_0.05: completed, 1.130 seconds, 372.0 mib peak


annual_0.02: completed, 3.036 seconds, 405.8 mib peak


annual_0.01: completed, 8.979 seconds, 417.5 mib peak


annual_0.005: completed, 25.664 seconds, 398.2 mib peak


,min_support,itemsets,rules,time_seconds,peak_memory_mib,outcome
0,0.050,191,282,1.130,371.969,completed
1,0.020,658,3159,3.036,405.766,completed
2,0.010,2267,9736,8.979,417.500,completed
3,0.005,7106,28433,25.664,398.250,completed


the boolean table itself is small enough, about 93.46 mib. default apriori makes a much larger
temporary table for the candidate combinations across all baskets.
at support 0.05 the failed allocation alone is about 4.76 gib.
low_memory=true evaluates candidates in smaller groups, so it avoids this large allocation.
it trades some speed for memory. i did not replace apriori with a different algorithm.

In [6]:
coverage_comparison = []
for run in experiments:
    kept = [r for r in run["rules"] if len(r["consequents"]) == 1 and r["lift"] > 1]
    covered = {r["consequents"][0] for r in kept}
    coverage_comparison.append({
        "min_support": run["min_support"],
        "saved_style_rules": len(kept),
        "recommendable_products": len(covered),
        "coverage": len(covered) / len(items)
    })
display(pd.DataFrame(coverage_comparison))
selected = experiments[-1]
raw_rules = pd.DataFrame(selected["rules"])

,min_support,saved_style_rules,recommendable_products,coverage
0,0.050,173,27,0.275510
1,0.020,1264,57,0.581633
2,0.010,3460,58,0.591837
3,0.005,11059,58,0.591837


### duplicate-product merge sensitivity

this is an additional experiment with the same million baskets, not a replacement for the four required runs.
i combine the two bread flags with or, remove the rustic column and keep 97 model columns.
the source catalogue and the main application still have 98 entries.
the merged name means a hypothetical baguette family, not a verified stock item or a new price.

In [7]:
merged_items = [item for item in items if item != "Pain baguette rustique"]
merged_values = values[:, [items.index(item) for item in merged_items]].copy()
merged_values[:, merged_items.index("Baguette")] = (
    values[:, items.index("Baguette")] | values[:, items.index("Pain baguette rustique")]
)
assert merged_values.shape == (1_000_000, 97)
assert int(merged_values[:, merged_items.index("Baguette")].sum()) == duplicate_audit["union_baskets"]
assert int(merged_values.sum()) == duplicate_audit["merged_presences"]
assert values.shape == (1_000_000, 98)
np.save(scratch / "merged.npy", merged_values)
(scratch / "merged_items.json").write_text(json.dumps(merged_items, ensure_ascii=False), encoding="utf-8")
merged_run = run_mining(
    "baguette_merge_sensitivity", scratch / "merged.npy", 0.005, True, scratch / "merged_items.json"
)
assert merged_run["outcome"] == "completed"
merged_rules = [
    rule for rule in merged_run["rules"]
    if len(rule["consequents"]) == 1 and rule["lift"] > 1
]
merged_covered = {rule["consequents"][0] for rule in merged_rules}
merged_sensitivity = {
    key: value for key, value in merged_run.items() if key not in {"rules", "itemsets"}
}
merged_sensitivity.update({
    "product_count": len(merged_items),
    "denominator": len(merged_items),
    "saved_style_rules": len(merged_rules),
    "recommended_products": len(merged_covered),
    "coverage": len(merged_covered) / len(merged_items)
})
display(pd.DataFrame([
    {
        "interpretation": "original records",
        "products": 98,
        "itemsets": selected["n_itemsets"],
        "raw_rules": selected["n_rules"],
        "time_seconds": round(selected["elapsed_seconds"], 3),
        "peak_memory_mib": round(selected["peak_memory_mib"], 3),
        "recommendable_products": coverage_comparison[-1]["recommendable_products"],
        "coverage_denominator": 98
    },
    {
        "interpretation": "hypothetical merged family",
        "products": 97,
        "itemsets": merged_run["n_itemsets"],
        "raw_rules": merged_run["n_rules"],
        "time_seconds": round(merged_run["elapsed_seconds"], 3),
        "peak_memory_mib": round(merged_run["peak_memory_mib"], 3),
        "recommendable_products": len(merged_covered),
        "coverage_denominator": 97
    }
]))
fondue_raw_key = (("Pain baguette rustique",), ("Bouillon à fondue",))
fondue_merged_key = (("Baguette",), ("Bouillon à fondue",))
raw_keys = {(tuple(rule["antecedents"]), tuple(rule["consequents"])) for rule in selected["rules"]}
merged_keys = {(tuple(rule["antecedents"]), tuple(rule["consequents"])) for rule in merged_run["rules"]}
assert fondue_raw_key in raw_keys
assert fondue_merged_key not in merged_keys
print("the rustic baguette to fondue broth rule exists before merging, but fails confidence after merging")
del merged_values, merged_run, merged_rules, raw_keys, merged_keys
gc.collect()

baguette_merge_sensitivity: completed, 25.652 seconds, 441.3 mib peak


,interpretation,products,itemsets,raw_rules,time_seconds,peak_memory_mib,recommendable_products,coverage_denominator
0,original records,98,7106,28433,25.664,398.250,58,98
1,hypothetical merged family,97,7142,28482,25.652,441.344,58,97


the rustic baguette to fondue broth rule exists before merging, but fails confidence after merging


0

the merge counts 108,610 bread baskets. adding the two product counts would double-count 3,009 baskets.
the bread records also have different behaviour: regular baguette is almost flat across seasons,
while rustic baguette appears about 3.2 times more often in winter than summer.
rustic baguette to fondue broth has about 41.00 percent confidence. after merging the confidence falls
to about 17.86 percent and this rule is lost at our 20 percent threshold.

the decision is to retain both catalogue records. the merge experiment shows a real loss of useful detail,
and the catalogue does not confirm that the two names are one stock item. different behaviour is evidence
for keeping the records, not proof of physical identity. the supplied catalogue has no confirmed duplicate identity. i document this negative result and keep
the original products.

## phase 3: evaluate the rules

support is the share of all baskets containing both sides of a rule together.

confidence is the share of baskets containing the left side that also contain the right side.

lift says how many times more likely the right side is with the left side than in an average basket.

i compare the same one-product to one-product rules before the lift filter.
the confidence threshold stays at 0.20 for both lists.

In [8]:
simple = raw_rules[
    raw_rules["antecedents"].map(len).eq(1) & raw_rules["consequents"].map(len).eq(1)
].copy()

def rule_table(frame):
    return pd.DataFrame({
        "if basket contains": frame["antecedents"].map(lambda group: ", ".join(names[item] for item in group)),
        "then suggest": frame["consequents"].map(lambda group: ", ".join(names[item] for item in group)),
        "support": frame["support"].to_numpy(),
        "confidence": frame["confidence"].to_numpy(),
        "lift": frame["lift"].to_numpy()
    }).reset_index(drop=True)

confidence_top = simple.sort_values(["confidence", "lift", "support"], ascending=False).head(10)
lift_top = simple.sort_values(["lift", "confidence", "support"], ascending=False).head(10)
print("top 10 by confidence")
display(rule_table(confidence_top))
print("top 10 by lift")
display(rule_table(lift_top))
bag_top_count = int(confidence_top["consequents"].map(lambda group: group == ["Sac réutilisable"]).sum())
bag_rules = simple[simple["consequents"].map(lambda group: group == ["Sac réutilisable"])]
print(f"{bag_top_count} of the confidence top 10 recommend the reusable bag")
print(f"the bag baseline is {item_supports['Sac réutilisable']:.4%}")
print(f"bag-rule lift ranges from {bag_rules['lift'].min():.4f} to {bag_rules['lift'].max():.4f}")

top 10 by confidence


,if basket contains,then suggest,support,confidence,lift
0,cucumber,reusable bag,0.062853,0.902619,1.002881
1,ice cider,reusable bag,0.052520,0.902034,1.002231
2,tomato sauce,reusable bag,0.095588,0.901808,1.001980
3,relish,reusable bag,0.053301,0.901695,1.001854
4,olive oil,reusable bag,0.049201,0.901662,1.001818
5,apples,reusable bag,0.082351,0.901617,1.001768
6,dijon mustard,reusable bag,0.074374,0.901525,1.001665
7,swiss cheese,reusable bag,0.083997,0.901458,1.001592
8,beef steak,reusable bag,0.095092,0.901414,1.001543
9,avocado,reusable bag,0.039907,0.901344,1.001464


top 10 by lift


,if basket contains,then suggest,support,confidence,lift
0,raclette cheese,fondue broth,0.033651,0.491198,10.170786
1,fondue broth,raclette cheese,0.033651,0.696780,10.170786
2,deli meats,raclette cheese,0.031932,0.683403,9.975519
3,raclette cheese,deli meats,0.031932,0.466106,9.975519
4,raclette cheese,pickles,0.029733,0.434008,9.772967
5,pickles,raclette cheese,0.029733,0.669526,9.772967
6,rustic baguette,raclette cheese,0.027627,0.654374,9.551784
7,raclette cheese,rustic baguette,0.027627,0.403267,9.551784
8,nori seaweed,sushi rice,0.054431,0.795019,9.504685
9,sushi rice,nori seaweed,0.054431,0.650738,9.504685


10 of the confidence top 10 recommend the reusable bag
the bag baseline is 90.0026%
bag-rule lift ranges from 0.9965 to 1.0029


my prediction was correct. the confidence list is dominated by the reusable bag.
its high confidence is close to the bag's already high baseline.
the lift list instead shows more specific basket links, mainly raclette meals and sushi products.
raclette cheese to fondue broth has about 49.12 percent confidence and 10.17 lift.
this is much more increase over the ordinary popularity than the bag rules, even with lower confidence.

i keep lift above one because it gives a positive association over the baseline.
this filter alone does not mean a rule is strong or useful. a lift just above one can still be weak,
so the application also shows its value and a strength label.

i choose support 0.005 for the saved model. it requires at least 5,000 baskets and keeps more combinations
than the higher thresholds. confidence 0.20 removes very low probability suggestions.
coverage is the same 58 products at 0.01 and 0.005. i accept the extra mining cost at 0.005 to keep
more context-specific combinations; it does not fix missing-product coverage.
only single-product consequents are exported,
because each application card recommends one product; antecedents can contain several products.

In [9]:
filtered_rules = [
    rule for rule in selected["rules"]
    if len(rule["consequents"]) == 1 and rule["lift"] > 1
]
filtered_rules.sort(
    key=lambda rule: (
        -rule["lift"], -rule["confidence"], -rule["support"],
        rule["consequents"], rule["antecedents"]
    )
)
covered_items = {rule["consequents"][0] for rule in filtered_rules}
coverage = len(covered_items) / len(items)
print(f"{len(filtered_rules):,} saved rules")
print(f"{len(covered_items)} of {len(items)} products can be recommended: {coverage:.2%} coverage")
print(f"{sum(len(rule['antecedents']) > 1 for rule in filtered_rules):,} rules have several triggering products")
rare = frequency.tail(5).sort_values("baskets")
display(rare.reset_index(drop=True))
print("these five rare products are below 5,000 baskets, so no saved rule can contain them")
diaper_beer = simple[
    simple["antecedents"].map(lambda group: group == ["Couches"])
    & simple["consequents"].map(lambda group: group == ["Bière"])
]
print("the retail example: diapers to beer")
display(rule_table(diaper_beer))

11,059 saved rules
58 of 98 products can be recommended: 59.18% coverage
10,705 rules have several triggering products


,product,baskets,support
0,foie gras,756,0.000756
1,aged balsamic vinegar,779,0.000779
2,caviar,810,0.000810
3,truffle oil,817,0.000817
4,saffron,818,0.000818


these five rare products are below 5,000 baskets, so no saved rule can contain them
the retail example: diapers to beer


,if basket contains,then suggest,support,confidence,lift
0,diapers,beer,0.030696,0.365577,2.429597


diapers and beer is the famous retail example reproduced in this data.
the association does not show that buying diapers causes someone to buy beer.
we do not have a controlled experiment and other customer habits can explain the link.
it is a pattern in this supplied transaction file, not a causal claim about real shoppers.

## phase 4: seasonal analysis

summer means june, july and august. winter means december, january and february from the supplied year.
the year is 2026, so this winter group combines the two ends of that year rather than one continuous winter.
i use the same support 0.005, confidence 0.20 and lift above one for both groups.

In [10]:
season_runs = {}
season_counts = {}
season_masks = {
    "summer": np.isin(months, [6, 7, 8]),
    "winter": np.isin(months, [12, 1, 2])
}
for season, mask in season_masks.items():
    season_values = values[mask]
    season_counts[season] = len(season_values)
    path = scratch / f"{season}.npy"
    np.save(path, season_values)
    del season_values
    season_runs[season] = run_mining(season, path, 0.005, True)
display(pd.DataFrame([{
    "season": season,
    "baskets": season_counts[season],
    "itemsets": run["n_itemsets"],
    "rules": run["n_rules"],
    "time_seconds": round(run["elapsed_seconds"], 3),
    "peak_memory_mib": round(run["peak_memory_mib"], 3)
} for season, run in season_runs.items()]))

summer: completed, 9.893 seconds, 308.5 mib peak


winter: completed, 5.268 seconds, 209.8 mib peak


,season,baskets,itemsets,rules,time_seconds,peak_memory_mib
0,summer,252474,9867,64011,9.893,308.484
1,winter,246965,6042,18348,5.268,209.766


In [11]:
item_index = {item: i for i, item in enumerate(items)}

def rule_key(rule):
    return (tuple(sorted(rule["antecedents"])), tuple(sorted(rule["consequents"])))

season_rule_keys = {
    season: {
        rule_key(rule) for rule in run["rules"]
        if len(rule["consequents"]) == 1 and rule["lift"] > 1
    }
    for season, run in season_runs.items()
}
season_raw_keys = {
    season: {rule_key(rule) for rule in run["rules"]}
    for season, run in season_runs.items()
}

def season_metrics(antecedents, consequent, season):
    mask = season_masks[season]
    left = values[:, [item_index[item] for item in antecedents]].all(axis=1) & mask
    right = values[:, item_index[consequent]] & mask
    joint = int((left & right).sum())
    n_left = int(left.sum())
    n_right = int(right.sum())
    total = int(mask.sum())
    confidence = joint / n_left if n_left else None
    lift = confidence / (n_right / total) if n_left and n_right else None
    key = (tuple(sorted(antecedents)), (consequent,))
    return {
        "season": season,
        "if basket contains": ", ".join(names[item] for item in antecedents),
        "then suggest": names[consequent],
        "baskets_together": joint,
        "support": joint / total,
        "confidence": confidence,
        "lift": lift,
        "in_mined_rules": key in season_raw_keys[season],
        "in_filtered_rules": key in season_rule_keys[season]
    }

season_examples = [
    (["Charbon de bois"], "Steak de boeuf"),
    (["Charbon de bois"], "Pain à hamburger"),
    (["Fromage à raclette"], "Charcuteries"),
    (["Couches"], "Bière")
]
season_comparison = pd.DataFrame([
    season_metrics(left, right, season)
    for left, right in season_examples
    for season in ["summer", "winter"]
])
display(season_comparison)
disappearing = next(
    rule for rule in season_runs["summer"]["rules"]
    if set(rule["antecedents"]) == {"Steak de boeuf", "Pain à hamburger", "Ketchup"}
    and rule["consequents"] == ["Relish"]
)
assert rule_key(disappearing) not in season_raw_keys["winter"]
disappearing_comparison = pd.DataFrame([
    season_metrics(disappearing["antecedents"], disappearing["consequents"][0], season)
    for season in ["summer", "winter"]
])
print("a rule present in summer and absent from the winter mined rules")
display(disappearing_comparison)

,season,if basket contains,then suggest,baskets_together,support,confidence,lift,in_mined_rules,in_filtered_rules
0,summer,charcoal,beef steak,47297,0.187334,0.694808,3.498625,True,True
1,winter,charcoal,beef steak,6210,0.025145,0.509016,12.660815,True,True
2,summer,charcoal,burger buns,44735,0.177187,0.657172,3.487887,True,True
3,winter,charcoal,burger buns,5816,0.023550,0.476721,12.266460,True,True
4,summer,raclette cheese,deli meats,1836,0.007272,0.265779,12.006126,True,True
5,winter,raclette cheese,deli meats,15812,0.064025,0.525403,6.718242,True,True
6,summer,diapers,beer,7527,0.029813,0.369714,2.019451,True,True
7,winter,diapers,beer,7146,0.028935,0.367801,2.541804,True,True


a rule present in summer and absent from the winter mined rules


,season,if basket contains,then suggest,baskets_together,support,confidence,lift,in_mined_rules,in_filtered_rules
0,summer,"ketchup, burger buns, beef steak",relish,6952,0.027536,0.354821,3.421156,True,True
1,winter,"ketchup, burger buns, beef steak",relish,877,0.003551,0.350800,12.990752,False,False


the table compares the same products, including when a rule is missing from a season's saved rules.
the three-product barbecue basket to relish rule is present in summer, with about 2.75 percent support,
but absent from winter mining because its support is about 0.36 percent, below the 0.50 percent threshold.
there are still 877 winter co-purchases. missing from the rule set does not mean zero co-purchases.

for charcoal to beef steak, summer support is about 18.73 percent and confidence is 69.48 percent,
while winter support is 2.51 percent and confidence is 50.90 percent.
however lift rises from about 3.50 to 12.66 because beef steak itself is much less common in winter.
the changing baseline matters: lower seasonal demand does not always mean lower lift.

i would rebuild the seasonal rules at least every month and check them before summer and winter campaigns.
the app in this submission uses the annual rules. in january these can overstate a barbecue buying probability
and give rankings based on the wrong seasonal baseline. some seasonal links are also diluted by annual averages.
a next version should load the current season's rule file.
monthly rebuilding is a practical starting choice, it should be reviewed with new transaction data.

## phase 5: save the offline model

the application needs no transaction file scan and does not call apriori.
the metadata records the real thresholds, date, counts, product coverage and popularity fallback.
strength labels are weak below 1.2, moderate from 1.2 to below 2, strong from 2 to below 4,
and very strong from 4. these are display choices, not statistical significance tests.

In [12]:
metadata = {
    "min_support": 0.005,
    "min_confidence": minimum_confidence,
    "min_lift_exclusive": 1.0,
    "low_memory": True,
    "max_len": None,
    "mining_date": datetime.now(ZoneInfo("America/Toronto")).isoformat(timespec="seconds"),
    "n_rules": len(filtered_rules),
    "raw_rule_count": selected["n_rules"],
    "basket_count": len(basket),
    "product_count": len(items),
    "recommended_products": len(covered_items),
    "coverage": coverage,
    "item_supports": {item: float(item_supports[item]) for item in items},
    "popularity": [
        {"item": item, "count": int(item_counts[item]), "support": float(item_supports[item])}
        for item in sorted(items, key=lambda item: (-item_counts[item], item))
    ],
    "strength_thresholds": {"moderate": 1.2, "strong": 2.0, "very_strong": 4.0},
    "season_basket_counts": season_counts,
    "python_version": platform.python_version(),
    "package_versions": {package: version(package) for package in ["numpy", "pandas", "mlxtend"]}
}
models = root / "models"
models.mkdir(exist_ok=True)
(models / "rules.json").write_text(
    json.dumps(filtered_rules, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
(models / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
print(f"saved {len(filtered_rules):,} rules, coverage {coverage:.2%}")

saved 11,059 rules, coverage 59.18%


## phase 6: measurements, limits and checks

coverage counts different products on the right side of the final rules, divided by the 98 catalogue products.
it does not count baskets with a match, and it does not count popularity fallback as rule coverage.
the confidence threshold and support pruning leave part of the catalogue without a rule.
popular-only recommendations can keep already popular products visible and hide the other products even more.

first limitation: rare and new products have too little support. i would add content-based candidates using
category and product details, with a small exploration slot, then measure their results.

second limitation: one annual model does not adapt to seasonal demand or individual customer preferences.
i would first use monthly rebuilt seasonal rules; customer history would need an extra personalised layer.

the rules show co-purchases, not customer satisfaction, profit or a causal effect.
the explanations and the fired-rule table let a person check why each suggestion was made.

In [13]:
def short_run(run):
    return {key: value for key, value in run.items() if key not in {"rules", "itemsets"}}

evidence = {
    "profile": profile,
    "default_run": short_run(default_run),
    "annual_runs": [short_run(run) for run in experiments],
    "coverage_comparison": coverage_comparison,
    "duplicate_audit": duplicate_audit,
    "merged_sensitivity": merged_sensitivity,
    "duplicate_rule_comparison": duplicate_rule_comparison,
    "confidence_top10": rule_table(confidence_top).to_dict("records"),
    "lift_top10": rule_table(lift_top).to_dict("records"),
    "bag_top10_count": bag_top_count,
    "bag_rule_lift_min": float(bag_rules["lift"].min()),
    "bag_rule_lift_max": float(bag_rules["lift"].max()),
    "diapers_beer": rule_table(diaper_beer).to_dict("records"),
    "season_runs": {season: short_run(run) for season, run in season_runs.items()},
    "season_comparison": season_comparison.to_dict("records"),
    "disappearing_rule": disappearing_comparison.to_dict("records"),
    "coverage": coverage,
    "recommended_products": len(covered_items),
    "saved_rules": len(filtered_rules),
    "multi_antecedent_rules": sum(len(rule["antecedents"]) > 1 for rule in filtered_rules),
    "uncovered_products": sorted(names[item] for item in set(items) - covered_items),
    "top10_products": frequency.head(10).to_dict("records"),
    "rarest5_products": rare.to_dict("records")
}
reports = root / "reports"
reports.mkdir(exist_ok=True)
(reports / "analysis_results.json").write_text(
    json.dumps(evidence, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
saved = json.loads((models / "rules.json").read_text(encoding="utf-8"))
assert len(saved) == metadata["n_rules"]
assert all(rule["lift"] > 1 and len(rule["consequents"]) == 1 for rule in saved)
assert all(set(rule["antecedents"]).issubset(items) for rule in saved)
assert len({rule["consequents"][0] for rule in saved}) == metadata["recommended_products"]
assert metadata["coverage"] == metadata["recommended_products"] / 98
print("model files and coverage checks passed")
workspace.cleanup()

model files and coverage checks passed
